# Agentic RAG: Autonomous Reasoning & Tool-Using Architecture Evaluation

This notebook benchmarks an **Agentic RAG** system equipped with specialized tools:
- `document_search`: Semantic vector retrieval.
- `calculator`: Deterministic mathematical evaluation.
- `comparative_analyzer`: Structured entity comparison.
- `table_data_analyzer`: Pandas DataFrame querying on CSV datasets.
- `document_summary`: Abstractive document synthesis.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from agentic_engine import AgentConfig, AgenticEngine

config = AgentConfig(
    data_dir=Path('sample_docs'),
    chroma_dir=Path('data/chroma_db')
)
agent = AgenticEngine(config)
print('Agent initialized on compute device:', agent.device)


## 1. Multi-Modal Document & Data Ingestion
Ingests text documents and structured CSV data into the unified knowledge base.


In [ ]:
report = agent.ingest_files()
print(f"Ingestion summary: {report['chunks']} text chunks across {report['files']} files.")
print(f"Active Tabular DataFrames: {list(agent.dataframes.keys())}")


## 2. Multi-Hop and Multi-Tool Query Evaluation
Evaluates the agent's planning and tool dispatching against `evaluation_cases.csv`.


In [ ]:
cases_df = pd.read_csv('evaluation_cases.csv')
eval_results = []

for _, row in cases_df.iterrows():
    q = row['question']
    q_type = row['query_type']
    trace = agent.run_agentic_pipeline(q)
    tools_invoked = [s['tool'] for s in trace['steps']]
    eval_results.append({
        'Question': q[:40] + '...',
        'Query Type': q_type,
        'Tools Invoked': ', '.join(tools_invoked),
        'Execution Time (s)': trace['execution_time'],
        'Sources Cited': len(trace['sources'])
    })

res_df = pd.DataFrame(eval_results)
res_df


## 3. Tool Usage Breakdown & Analytics
Visualizing how the autonomous agent dynamically dispatches tools based on query intent:


In [ ]:
all_tools = []
for tools in res_df['Tools Invoked']:
    all_tools.extend([t.strip() for t in tools.split(',')])

tool_counts = pd.Series(all_tools).value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
tool_counts.plot(kind='bar', color='#2563EB', edgecolor='black', ax=ax)
ax.set_title('Agent Tool Invocation Frequency across Complex Queries')
ax.set_xlabel('Specialized Tool')
ax.set_ylabel('Invocations')
plt.xticks(rotation=25, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
